In [1]:
import numpy as np
import my_code.diffusion_training_sign_corr.data_loading as data_loading
import yaml
from tqdm import tqdm
import metrics.geodist_metric as geodist_metric
from utils.shape_util import compute_geodesic_distmat
import torch



Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
dataset_name = 'SHREC19_r_pair'

single_dataset, pair_dataset = data_loading.get_val_dataset(
    dataset_name, 'test', 128, preload=False, return_evecs=True, centering='bbox'
)

dist_mat_list = []

for i in tqdm(range(len(single_dataset))):

    data_i = single_dataset[i]

    dist_mat = torch.tensor(
        compute_geodesic_distmat(data_i['verts'].numpy(), data_i['faces'].numpy())    
    )
    
    dist_mat_list.append(dist_mat)

100%|██████████████████████████████████████████████████████████████████████████| 44/44 [06:37<00:00,  9.03s/it]


In [ ]:
path = f'/home/s94zalek_hpc/DenoisingFunctionalMaps/results/to_remove/'\
        f'ddpm_96/SHREC19_r'

geo_err_list = []

for i in tqdm(range(len(pair_dataset))):
    
    data_i = pair_dataset[i]
    
    first_idx = data_i['first']['id']
    second_idx = data_i['second']['id']
    
    off_file_first = single_dataset.off_files[first_idx]
    off_file_second = single_dataset.off_files[second_idx]

    # from data/FAUST_r/off/tr_reg_080.off get tr_reg_080
    first_off_name = off_file_first.split('/')[-1].split('.')[0]
    second_off_name = off_file_second.split('/')[-1].split('.')[0]
    
    p2p = f'{path}/{first_off_name}_{second_off_name}.pt'   
    p2p = torch.load(p2p, weights_only=True)
        
    dist_x = dist_mat_list[first_idx]
    dist_y = dist_mat_list[second_idx]
    
    corr_first = data_i['first']['corr']
    corr_second = data_i['second']['corr']
    
    geo_err = geodist_metric.calculate_geodesic_error(
        dist_x, corr_first.cpu(), corr_second.cpu(), p2p, return_mean=True
    ) * 100
    
    # geo_err = geodist_metric.calculate_geodesic_error(
    #     dist_y, corr_second.cpu(), corr_first.cpu(), p2p, return_mean=True
    # ) * 100
    
    # print(first_off_name, second_off_name, geo_err)
    
    geo_err_list.append(geo_err)
    
    # break
    
geo_err_list = torch.tensor(geo_err_list)
print(f'{dataset_name}, mean geo err: {geo_err_list.mean():.1f}')

  0%|                                                                                  | 0/407 [00:00<?, ?it/s]

  0%|                                                                                  | 0/407 [00:00<?, ?it/s]


TypeError: can't convert cuda:0 device type tensor to numpy. Use Tensor.cpu() to copy the tensor to host memory first.

In [ ]:
import os

base_dir = '/home/s94zalek_hpc/DenoisingFunctionalMaps/results/ddpm_32 copy'

# for each subdirectory
for sub_dir in os.listdir(base_dir):
    # check if it is a directory
    
    if os.path.isdir(f'{base_dir}/{sub_dir}'):
        # check if the directory name contains 'ddpm'
        
        # rename the directory to p2p
        new_dir_name = 'p2p'
        os.rename(f'{base_dir}/{sub_dir}', f'{base_dir}/{new_dir_name}')
        
        # create a directory named sub_dir
        os.makedirs(f'{base_dir}/{sub_dir}', exist_ok=True)
        # move p2p directory to sub_dir
        
        os
        
    pass